# Part 4-B — RAG Implementation

**Components:**
- Vector DB: FAISS (`faiss-cpu`) — pure pip install, no server required
- Embedding model: `paraphrase-multilingual-MiniLM-L12-v2` (English + Turkish)
- LLM: Qwen 2.5 7B Instruct (4-bit quantized)

**Steps:**
1. Build FAISS index from WMT16 train corpus
2. Demo retrieval for sample queries
3. Run RAG translation on 200 test samples
4. Compute COMET score

> Index is cached to `data/faiss_index.*` — subsequent runs load instantly.

In [1]:
import sys
sys.path.insert(0, '..')

import os
import json
import numpy as np
import matplotlib.pyplot as plt

from modules.dataset import load_wmt16, get_samples, get_corpus
from modules.model import load_model
from modules.retrieval import (
    load_embedding_model,
    build_faiss_index,
    retrieve_examples,
    make_retriever,
)
from modules.translation import translate_with_rag, load_cached_translations
from modules.evaluation import load_comet_model, compute_comet

os.makedirs('../data', exist_ok=True)
os.makedirs('../results', exist_ok=True)

## 4-B.1 Load Corpus & Test Data

In [2]:
# Load train corpus for indexing (cap at 50K for speed; full 207K also works)
CORPUS_MAX = 50_000

print("Loading WMT16 train split for RAG corpus...")
train_ds = load_wmt16(split='train')
corpus_pairs = get_corpus(train_ds, max_size=CORPUS_MAX)
print(f"Corpus: {len(corpus_pairs)} pairs")

Loading WMT16 train split for RAG corpus...
Loading WMT16 tr-en [train] split...
  Loaded 205756 sentence pairs.
Corpus size: 50000 pairs.
Corpus: 50000 pairs


In [3]:
# Load test data (same 200 samples as Part 3 for comparability)
test_ds = load_wmt16(split='test')
en_sentences, tr_references = get_samples(test_ds, n=200, seed=42)
print(f"Test samples: {len(en_sentences)}")

Loading WMT16 tr-en [test] split...
  Loaded 3000 sentence pairs.
Sampled 200 pairs (seed=42).
Test samples: 200


## 4-B.2 Embedding Generation & FAISS Index

In [4]:
# Load embedding model
emb_model = load_embedding_model()

Loading embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Embedding dim: 384


/root/WMT16_RAG/notebooks/../modules/retrieval.py:42: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"  Embedding dim: {model.get_sentence_embedding_dimension()}")


In [5]:
# Build or load cached FAISS index
INDEX_PATH = '../data/faiss_index'

index, corpus_pairs = build_faiss_index(
    corpus_pairs,
    emb_model,
    index_path=INDEX_PATH
)

print(f"\nFAISS index stats:")
print(f"  Total vectors : {index.ntotal:,}")
print(f"  Dimension     : {index.d}")
print(f"  Index type    : {type(index).__name__} (exact cosine search)")

Building FAISS index for 50000 pairs...


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

  Index built: 50000 vectors, dim=384
  Index saved to ../data/faiss_index.*

FAISS index stats:
  Total vectors : 50,000
  Dimension     : 384
  Index type    : IndexFlatIP (exact cosine search)


## 4-B.3 Similarity Search Demo

In [6]:
# Demo: retrieve top-5 examples for 3 queries
demo_queries = [
    "The parliament approved the new budget plan.",
    "Scientists discovered a new species in the Amazon rainforest.",
    "The company reported record profits in the third quarter.",
]

for query in demo_queries:
    examples = retrieve_examples(query, index, corpus_pairs, emb_model, k=3)
    print(f"Query: {query}")
    print("Top-3 retrieved examples:")
    for i, (en, tr) in enumerate(examples, 1):
        en_disp = en[:70] + '...' if len(en) > 70 else en
        tr_disp = tr[:70] + '...' if len(tr) > 70 else tr
        print(f"  [{i}] EN: {en_disp}")
        print(f"       TR: {tr_disp}")
    print()

Query: The parliament approved the new budget plan.
Top-3 retrieved examples:
  [1] EN: Parliament is expected to approve the new cabinet Monday with the supp...
       TR: Parlamentonun yeni kabineyi Pazartesi günü Düzen, Hukuk ve Güvenlik pa...
  [2] EN: Parliament endorsed the national development plan on Wednesday (28 Jun...
       TR: Meclis, ulusal kalkınma planını 28 Haziran Çarşamba günü kabul etti. [...
  [3] EN: Macedonia's parliament approved an updated version of the country's 20...
       TR: Makedonya Parlamentosu 2007 bütçesinin güncellenmiş halini onayladı.

Query: Scientists discovered a new species in the Amazon rainforest.
Top-3 retrieved examples:
  [1] EN: Scientists from the Bulgarian Museum of Nature and Science discovered ...
       TR: Bulgaristan Doğa ve Bilim Müzesi'ne bağlı bilim insanları 5,5 milyon y...
  [2] EN: Also in science news: Bulgarian scientists find a prehistoric deer ske...
       TR: Bilim haberlerinde ayrıca: Bulgar bilim insanları tarih önce

## 4-B.4 Integration with LLM Pipeline

In [7]:
# Show a full RAG prompt for one example
from modules.prompts import rag_prompt

sample_query = en_sentences[0]
examples = retrieve_examples(sample_query, index, corpus_pairs, emb_model, k=5)
prompt = rag_prompt(sample_query, examples, direction='en->tr')

print("=" * 60)
print("FULL RAG PROMPT (sample):")
print("=" * 60)
print(prompt)
print("=" * 60)

FULL RAG PROMPT (sample):
You are an expert English-to-Turkish translator. Here are 5 similar translation examples for reference:

Example 1:
English: "These cowardly murderers will be brought to justice," he added, stressing that such incidents will not deter police from efforts "to keep our citizens, neighbourhoods and cities safe".
Turkish: "Bu korkak katiller adalete teslim edilecektir." diyen bakan, bu gibi olayların polisi "vatandaşlarımızı, mahallelerimizi ve kentlerimizi emniyette tutma" çabalarından caydırmayacağını da vurguladı.

Example 2:
English: If proven, those who mistreated the suspects would be punished, she said.
Turkish: Yetkili, kanıtlandığı takdirde zanlılara kötü muamele yapanların cezalandırılacağını söyledi.

Example 3:
English: "The victim should not leave the house, the violent person should leave," she said.
Turkish: Egro, "Evi mağdur kişi değil, şiddeti uygulayan kişi terk etmeli." diyor.

Example 4:
English: When you try to tell their stories, you eventual

In [8]:
# Load LLM
model, tokenizer = load_model(quantize=True)

Loading tokenizer: Qwen/Qwen2.5-7B-Instruct
Using 4-bit NF4 quantization (bitsandbytes)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/root/WMT16_RAG/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model ready.


## 4-B.5 RAG Translation — 200 Test Samples

In [9]:
RAG_PATH = '../results/rag_translations.json'

# Create retriever closure (k=5 examples per query)
retriever = make_retriever(index, corpus_pairs, emb_model, k=5)

if os.path.exists(RAG_PATH):
    print(f"Loading cached RAG translations from {RAG_PATH}")
    rag_translations = load_cached_translations(RAG_PATH)
else:
    rag_translations = translate_with_rag(
        model, tokenizer, en_sentences,
        retriever=retriever,
        direction='en->tr',
        save_path=RAG_PATH
    )

print(f"\nSample RAG outputs:")
for i in range(3):
    print(f"  EN : {en_sentences[i]}")
    print(f"  REF: {tr_references[i]}")
    print(f"  RAG: {rag_translations[i]}")
    print()

[RAG] Translating:   0%|          | 0/200 [00:00<?, ?it/s]/root/WMT16_RAG/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[RAG] Translating: 100%|██████████| 200/200 [11:22<00:00,  3.41s/it]

Saved 200 translations → ../results/rag_translations.json

Sample RAG outputs:
  EN : How can they allow their conscience to stand by her killers?
  REF: Onun katillerinin yanında yer almayı bunlar nasıl vicdanlarına sığdırıyorlar?
  RAG: Onların akla uygun olmamalarına nasıl izin verebilirler onların katillerine?

  EN : Among the dead was father-of-four Qasim Akram, from Bolton, Greater Manchester, who was on his first pilgrimage when the crane crashed down.
  REF: Ölüler arasında vinç düştüğünde ilk haccında olan Manchester'ın Bolton bölgesinden dört çocuk babası Kasım Akram da vardı.
  RAG: Dört çocuk babası Kasım Akram, Bolton'tan Büyük Manchester, kran çarptığında ilk gezi sırasında vefat etti.

  EN : And these numbers hold up in early states.
  REF: Ve bu sayılar erken evrelerde yüksekte seyrediyor.
  RAG: Ve bu sayılar erken devletlerde geçerli kalır.



## 4-B.6 COMET Evaluation

In [ ]:
comet_model = load_comet_model()

print("Computing COMET — RAG (5-shot)...")
rag_result = compute_comet(en_sentences, rag_translations, tr_references, comet_model, gpus=0)

rag_scores = {'rag_5shot': rag_result['system_score']}

# Save for Part 5
with open('../results/part4_comet_scores.json', 'w') as f:
    json.dump(rag_scores, f, indent=2)

print(f"\nRAG 5-shot COMET: {rag_result['system_score']:.4f}")

/root/WMT16_RAG/.venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Loading COMET model: Unbabel/wmt22-comet-da


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/root/WMT16_RAG/.venv/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning E

COMET model ready.
Computing COMET — RAG (5-shot)...


OutOfMemoryError: CUDA out of memory. Tried to allocate 72.00 MiB. GPU 0 has a total capacity of 22.03 GiB of which 53.69 MiB is free. Process 6096 has 218.00 MiB memory in use. Process 15361 has 6.89 GiB memory in use. Process 16545 has 6.88 GiB memory in use. Including non-PyTorch memory, this process has 7.97 GiB memory in use. Of the allocated memory 7.71 GiB is allocated by PyTorch, and 24.57 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 4-B.7 Retrieval Quality Analysis

In [ ]:
import faiss

# Compute similarity scores for the first 50 test queries
from modules.retrieval import embed_sentences

query_embs = embed_sentences(en_sentences[:50], emb_model, show_progress=False)
scores, _ = index.search(query_embs, 5)
top1_scores = scores[:, 0]  # highest similarity per query

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(top1_scores, bins=20, color='#2ecc71', edgecolor='white', alpha=0.9)
ax.set_xlabel('Top-1 Cosine Similarity Score', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Top-1 Retrieval Similarity Scores\n(first 50 test queries)', fontsize=12)
ax.axvline(top1_scores.mean(), color='red', linestyle='--',
           label=f'Mean: {top1_scores.mean():.3f}')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/retrieval_similarity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Mean top-1 similarity: {top1_scores.mean():.3f}")
print(f"Min: {top1_scores.min():.3f}  Max: {top1_scores.max():.3f}")